# Big Data with DuckDB

**What you'll learn:** How to use DuckDB to run fast SQL queries directly on CSV files, join datasets, and convert results to pandas.

**Prerequisites:** [01 - Python Basics](01-python-basics.ipynb), [02 - Big Data Intro](02-big-data-intro.ipynb)

**Before you start:** Make sure you've generated the large dataset:
```bash
python scripts/generate_large_data.py
```

## What is DuckDB?

DuckDB is an **in-process analytical database**. Let's unpack that:

- **In-process** — it runs inside your Python program, not as a separate server. No setup, no configuration.
- **Analytical** — it's optimized for queries that scan many rows and compute aggregations (sums, averages, counts). This is what data analysis mostly does.
- **Database** — it speaks SQL, the standard language for querying data.

Think of it as "SQLite for analytics." SQLite is great for apps that insert/update individual records; DuckDB is great for analyzing millions of records.

## Getting Started

Connecting to DuckDB takes one line:

In [ ]:
import duckdb

con = duckdb.connect()  # in-memory database
print("DuckDB is ready.")

That's it. No server to start, no credentials, no configuration.

## Querying CSV Files Directly

This is DuckDB's killer feature: you can run SQL queries on CSV files without loading them first.

In [ ]:
result = con.sql("SELECT * FROM '../data/sales.csv' LIMIT 5")
print(result)

DuckDB reads the CSV, figures out the column types, and runs the query — all in one step.

## SQL Basics

If you haven't used SQL before, here's a quick primer. SQL (Structured Query Language) is how you talk to databases. The main clauses are:

| Clause | Purpose | Example |
|--------|---------|----------|
| `SELECT` | Which columns to return | `SELECT product, quantity` |
| `FROM` | Where to read data from | `FROM '../data/sales.csv'` |
| `WHERE` | Filter rows | `WHERE quantity > 5` |
| `GROUP BY` | Group rows for aggregation | `GROUP BY category` |
| `ORDER BY` | Sort results | `ORDER BY total DESC` |
| `LIMIT` | Return only N rows | `LIMIT 10` |

### SELECT and WHERE

In [ ]:
con.sql("""
    SELECT product, category, unit_price, quantity
    FROM '../data/sales.csv'
    WHERE category = 'Electronics' AND unit_price > 500
    LIMIT 10
""")

### COUNT, SUM, AVG

In [ ]:
con.sql("""
    SELECT 
        COUNT(*) AS total_transactions,
        SUM(quantity) AS total_items_sold,
        ROUND(AVG(unit_price), 2) AS avg_price
    FROM '../data/sales.csv'
""")

### GROUP BY

In [ ]:
con.sql("""
    SELECT 
        category,
        COUNT(*) AS num_transactions,
        ROUND(AVG(unit_price), 2) AS avg_price,
        SUM(quantity * unit_price) AS total_revenue
    FROM '../data/sales_large.csv'
    GROUP BY category
    ORDER BY total_revenue DESC
""")

### GROUP BY with two columns

In [ ]:
con.sql("""
    SELECT 
        category,
        region,
        COUNT(*) AS transactions,
        ROUND(SUM(quantity * unit_price), 2) AS revenue
    FROM '../data/sales_large.csv'
    GROUP BY category, region
    ORDER BY category, revenue DESC
    LIMIT 15
""")

## Joins

A **join** combines two tables based on a shared column. Let's join our sales data with the employee data to see which regions have which employees:

In [ ]:
# First, let's look at both tables
print("Sales columns:")
con.sql("SELECT * FROM '../data/sales.csv' LIMIT 3")

In [ ]:
print("Employee columns:")
con.sql("SELECT * FROM '../data/employees.csv' LIMIT 3")

In [ ]:
# What's the total sales revenue per employee department city?
# (This is a creative join — in real work, you'd join on a shared key like employee_id)
con.sql("""
    SELECT 
        e.department,
        COUNT(DISTINCT e.name) AS num_employees,
        ROUND(AVG(e.salary), 2) AS avg_salary
    FROM '../data/employees.csv' AS e
    GROUP BY e.department
    ORDER BY avg_salary DESC
""")

## Converting to pandas

DuckDB results can be converted to pandas DataFrames for further analysis or plotting:

In [ ]:
df = con.sql("""
    SELECT category, 
           SUM(quantity * unit_price) AS revenue
    FROM '../data/sales_large.csv'
    GROUP BY category
    ORDER BY revenue DESC
""").df()  # .df() converts to a pandas DataFrame

print(type(df))
print(df)

## Querying pandas DataFrames with SQL

DuckDB can also query pandas DataFrames directly — useful when you already have data in pandas and want SQL's power:

In [ ]:
import pandas as pd

employees = pd.read_csv("../data/employees.csv")

# Query the pandas DataFrame with SQL!
con.sql("""
    SELECT department, 
           ROUND(AVG(salary), 2) AS avg_salary,
           COUNT(*) AS headcount
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""")

## Speed Comparison

Let's time DuckDB vs pandas on the same aggregation:

In [ ]:
import time

# pandas
pdf = pd.read_csv("../data/sales_large.csv")
start = time.time()
pandas_result = pdf.groupby(["category", "region"]).agg(
    revenue=("unit_price", "sum"),
    count=("transaction_id", "count")
)
pandas_time = time.time() - start

# DuckDB
start = time.time()
duck_result = con.sql("""
    SELECT category, region,
           SUM(unit_price) AS revenue,
           COUNT(*) AS count
    FROM '../data/sales_large.csv'
    GROUP BY category, region
""").df()
duck_time = time.time() - start

print(f"pandas:  {pandas_time:.3f}s")
print(f"DuckDB:  {duck_time:.3f}s")
print(f"\nDuckDB is often faster, even at 100K rows,")
print(f"because its engine is highly optimized for analytical queries.")

## When to Use DuckDB

**Choose DuckDB when:**
- You want to run SQL queries on CSV/Parquet files without loading them into memory first
- You're comfortable with SQL (or want to learn it)
- You need fast aggregations on a single machine
- You want to query pandas DataFrames with SQL

**Choose something else when:**
- Data is small enough for pandas (just use pandas)
- You need distributed computing across many machines (use PySpark)
- You prefer a pandas-like API over SQL (use Dask)

---

## Key Takeaways

- DuckDB is an in-process analytical database — no server needed.
- It can query CSV files directly with SQL (`SELECT * FROM 'file.csv'`).
- Results convert to pandas with `.df()`.
- It can even query pandas DataFrames using SQL.
- It's very fast for analytical queries, often faster than pandas.

---

**Next up:** [06 - Data Visualization](06-visualization.ipynb) — learn to create charts and plots with matplotlib and seaborn.